# Overland flow with the OLF model: Manning's roughness and steady sheet flow

MODFLOW 6 can route water *over the land surface* - not only through the aquifer - with its **overland-flow (OLF)** model. OLF is a MODFLOW 6 model in its own right - a sibling of the groundwater-flow (GWF), transport (GWT), energy (GWE), and particle-tracking (PRT) models, with its own grid, packages, and solution - not a package inside a groundwater model. It solves the two-dimensional **diffusive-wave** approximation of the shallow-water equations on a grid of cells, so it can represent sheet flow, ponding, and runoff.

This notebook builds a small OLF model - based on the MODFLOW 6 `test_olf_dis` autotest - on a flat plane with an **inactive block** the water has to flow around. Water enters at a fixed high stage along the west edge and leaves at a fixed low stage along the east edge, and the **Diffusive Wave (DFW)** package spreads it across the surface. We run it for several values of **Manning's roughness coefficient** `n` to see what `n` does - and does not - control.

**A few terms.** *Stage* is the water-surface elevation; the water *depth* is the stage minus the land-surface (bottom) elevation. The **diffusive-wave** approximation drives flow with the slope of the *water surface*, and **Manning's `n`** sets the flow resistance - a smooth surface has a small `n` (about 0.03), a vegetated one a larger `n` (0.1 or more). By the end you will build an OLF model, plot the steady water surface, and find that `n` controls the *discharge* while, with fixed-stage boundaries, the *stage* pattern does not depend on it.

## Set up

Import FloPy to build and run the model, NumPy for array math, and the paired `mf6_olf` module, which holds the functions that read the results and draw the water-surface map.

In [ ]:
import pathlib as pl

import flopy
import mf6_olf
import numpy as np

## Model parameters

The domain is a flat plane, 15 rows by 20 columns of 100 m cells. A block of cells in the interior is marked **inactive** (`idomain = 0`) - an obstacle, such as a building or a rock outcrop, that water must flow around. Fixed water-surface stages on the west and east edges drive flow across the plane, and we compare a few values of Manning's `n`. Cell ids are zero-based `(row, column)`; lengths are in meters and time in seconds.

In [ ]:
# Grid: a flat plane of 100 m cells.
nrow, ncol = 15, 20
delr = delc = 100.0
bottom = np.zeros((nrow, ncol))  # flat land surface at elevation 0

# Mark an interior block inactive - an obstacle the flow must go around.
idomain = np.ones((nrow, ncol), dtype=int)
idomain[6:9, 8:12] = 0

# Fixed water-surface stages that drive flow from the west edge to the east edge.
inflow_stage = 2.0  # west edge (m)
outflow_stage = 1.0  # east edge (m)

# Manning's roughness values to compare.
manning_values = [0.03, 0.05, 0.10]

## Build the model

Build the whole simulation in one function so the runs differ only by Manning's `n`. An OLF model is assembled much like a groundwater-flow model, but with OLF packages: a 2-D structured grid (**DIS2D**), the **Diffusive Wave** flow package (**DFW**, which carries Manning's `n`), initial stage (**IC**), fixed-stage boundaries (**CHD**), and output control (**OC**). Because the diffusive-wave equation is nonlinear (flow depends on the water depth), the solver (**IMS**) uses under-relaxation to converge.

In [ ]:
def build_olf_simulation(ws, manningsn):
    """Build the steady overland-flow simulation for a given Manning's n."""
    name = "mf6-olf"
    sim = flopy.mf6.MFSimulation(sim_name=name, sim_ws=str(ws), exe_name="mf6")
    flopy.mf6.ModflowTdis(sim, nper=1, perioddata=[(1.0, 1, 1.0)], time_units="seconds")
    flopy.mf6.ModflowIms(
        sim,
        print_option="summary",
        outer_maximum=200,
        inner_maximum=100,
        outer_dvclose=1e-7,
        inner_dvclose=1e-7,
        under_relaxation="dbd",
        under_relaxation_theta=0.9,
        under_relaxation_kappa=0.0001,
        linear_acceleration="bicgstab",
    )
    olf = flopy.mf6.ModflowOlf(sim, modelname=name, save_flows=True)
    flopy.mf6.ModflowOlfdis2D(
        olf,
        nrow=nrow,
        ncol=ncol,
        delr=delr,
        delc=delc,
        bottom=bottom,
        idomain=idomain,
    )
    # DFW: the diffusive-wave flow package. length_conversion / time_conversion
    # put Manning's equation (SI: meters, seconds) in the model's length and
    # time units.
    flopy.mf6.ModflowOlfdfw(
        olf,
        manningsn=manningsn,
        save_flows=True,
        length_conversion=1.0,
        time_conversion=1.0,
    )
    flopy.mf6.ModflowOlfic(olf, strt=inflow_stage)
    # Fixed-stage cells along the west (inflow) and east (outflow) edges.
    chd = [[(i, 0), inflow_stage] for i in range(nrow)]
    chd += [[(i, ncol - 1), outflow_stage] for i in range(nrow)]
    flopy.mf6.ModflowOlfchd(olf, stress_period_data=chd)
    flopy.mf6.ModflowOlfoc(
        olf,
        budget_filerecord=f"{name}.bud",
        stage_filerecord=f"{name}.stage",
        saverecord=[("STAGE", "ALL"), ("BUDGET", "ALL")],
    )
    return sim

## Run the model for each Manning's n

Build, write, and run the model once per roughness value. From each run, read the total discharge across the plane as the inflow through the west (constant-stage) boundary.

In [ ]:
sims = {}
discharge = {}
for n in manning_values:
    ws = pl.Path("./models") / "mf6-olf" / f"n{n}"
    sim = build_olf_simulation(ws, n)
    sim.write_simulation(silent=True)
    success, buff = sim.run_simulation(silent=True)
    assert success, f"n={n} did not converge"
    sims[n] = sim
    discharge[n] = mf6_olf.total_discharge(sim)
    print(f"n = {n:4.2f}: discharge across the plane = {discharge[n]:8.1f} m^3/s")

## The steady water surface

Read the stage (water-surface elevation) from the base run and map it, masking the inactive block. The reading and plotting are done by helper functions in the paired module `mf6_olf.py`; the stage contours trace the water surface as it falls from the west edge to the east edge and bends around the obstacle.

In [ ]:
# The stage-reading and plotting helpers live in the paired module mf6_olf.py.
mf6_olf.plot_stage(
    sims[manning_values[0]],
    idomain,
    delr,
    delc,
    levels=np.linspace(outflow_stage, inflow_stage, 11),
)

**What to look for.** Water flows from the high-stage west edge (2 m) to the low-stage east edge (1 m), so the surface tilts steadily downhill in that direction. Around the grey inactive block the contours bend - the flow has to go around it, just as overland flow routes around a building or a rise in the ground. The contours crowd together downstream of the block, where the flow squeezes past.

## What Manning's n controls

Compare the runs. Discharge is the total water moving across the plane; the stage range is the span of the water surface. Watch the last two columns.

In [ ]:
print(
    f"{'Manning n':>10}{'discharge (m3/s)':>18}{'n x discharge':>15}{'stage min/max (m)':>22}"
)
for n in manning_values:
    st = mf6_olf.get_stage(sims[n], idomain)
    print(
        f"{n:>10.3f}{discharge[n]:>18.1f}{n * discharge[n]:>15.1f}"
        f"{float(st.min()):>13.2f} /{float(st.max()):>6.2f}"
    )

**What to look for.** Two things stand out. First, `n x discharge` is essentially **constant**, so the discharge is **inversely proportional to Manning's `n`** - doubling the roughness halves the flow, exactly as Manning's equation predicts. Second, the **stage** min and max (and, in fact, the whole water surface) are **identical** for every `n`. With fixed-stage boundaries and a flat bottom, the shape of the water surface is set by the geometry, and `n` only controls *how much* water moves down that surface. (Add a bottom slope or a fixed *inflow rate* instead of a fixed stage, and `n` would control the water depth as well.)

## Exercise

- Replace the fixed east-edge stage with a **free outflow**: add a bottom slope (make `bottom` decrease from west to east), drive the inflow with the **`ModflowOlfflw`** (inflow) package on the west edge, and let water leave through the **`ModflowOlfzdg`** (zero-depth-gradient) package on the east edge. Now how does Manning's `n` change the water *depth*?
- Add uniform rainfall over the whole surface with the **`ModflowOlfpcp`** (precipitation) package and watch where water accumulates.
- Move or resize the inactive block, or add a second one, and see how the flow reroutes.

Predict the effect first, then check it against the water-surface map and the discharge table.

## Recap

In this notebook you:

- built a two-dimensional **overland-flow (OLF)** model with the MODFLOW 6 **Diffusive Wave (DFW)** package, adapted from the `test_olf_dis` example, with fixed-stage inflow and outflow edges and an inactive block to flow around;
- ran it for several values of **Manning's roughness `n`** and mapped the steady water surface as it falls across the plane and bends around the obstacle; and
- found that discharge is **inversely proportional to `n`**, while the fixed-boundary water-surface stage is **independent of `n`** - a compact illustration of what roughness does in the diffusive-wave equation.